# HanziGen - 字型生成训练（本地机版）

> 本 notebook 适用于**在自己的电脑上**运行训练，无需任何云平台。
>
> 与云端版（`hanzigen_cloudstudio.ipynb` / `hanzigen_colab.ipynb` / `hanzigen_moda.ipynb`）的区别：
> - **硬件档位用 conservative（本地稳妥档）**：workers 留 2 核余量、预取保守、显存预留更稳，避免影响你日常使用、防止 OOM。
> - **内置硬件适配性检测**：Cell 1 会自动判断你的电脑是否满足训练要求（需要 NVIDIA GPU 或 Apple Silicon；AMD 核显/纯 CPU 会提示无法训练）。
> - 无需挂载网盘、无需 clone，**直接在当前项目目录运行**。

## 前置要求

1. 已安装 Python 3.10+ 与依赖：`pip install -r requirements.txt`
2. 有可用的训练加速器：**NVIDIA GPU（建议显存 >= 8GB）** 或 **Apple Silicon (MPS)**
   - 仅 AMD 核显 / 纯 CPU 的机器**无法运行训练**（本项目模型依赖 CUDA/MPS）
3. 目标字体 `.ttf` / `.otf` 已放入 `fonts/` 目录

## 流程

```
Cell 0: 配置字体名 + 阶段开关
Cell 1: 环境自检 + 硬件适配性检测 + 断连自检（可重复运行）
Cell 2: 数据准备（分析字体 → 生成数据集 → 提取字集）
Cell 3: 训练 VQ-VAE（conservative 档）
Cell 4: 训练 LDM（conservative 档）
Cell 5: 推理 + 指标 + 转 SVG
```

---
## Cell 0: 配置参数

> **只改这里！** 填你放入 `fonts/` 的字体文件名。

In [ ]:
# ==================== 修改你的字体文件名 ====================
TARGET_FONT = "myfont.ttf"    # 改成你放入 fonts/ 的字体名
# ==========================================================

FONT_NAME = TARGET_FONT.rsplit(".", 1)[0]

# ==================== 训练阶段开关（默认全开）====================
DO_DATA_PREP = True       # Cell 2: 数据准备
DO_TRAIN_VQVAE = True     # Cell 3: 训练 VQ-VAE
DO_TRAIN_LDM = True       # Cell 4: 训练 LDM
DO_INFERENCE = True       # Cell 5: 推理+指标+转SVG
# ==========================================================

STATE_FILE = "colab_state.json"

print(f"目标字体: {TARGET_FONT}")
print(f"字体名称: {FONT_NAME}")
print(f"阶段开关: 数据准备={DO_DATA_PREP} VQVAE={DO_TRAIN_VQVAE} LDM={DO_TRAIN_LDM} 推理={DO_INFERENCE}")

---
## Cell 1: 环境自检 + 硬件适配性检测 + 断连自检

> 会先检测你的电脑配置是否满足训练要求，再改写续训参数。可重复运行（幂等）。

In [ ]:
import os, json, re, glob
import torch
from utils.hardware import check_training_viability, detect_hardware

print("===== 硬件适配性检测 =====")
info = detect_hardware()
print(f"  CPU 核数: {info['cpu_cores']}")
if info["gpu_available"]:
    print(f"  GPU: {info['gpu_name']} ({info['vram_gb']:.1f} GB)")
elif info["mps_available"]:
    print(f"  GPU: {info['gpu_name']}（Apple Silicon MPS）")
else:
    print("  GPU: 未检测到（仅 CPU / 核显）")
print(f"  PyTorch: {torch.__version__} | CUDA: {torch.version.cuda or '无'}")

viability = check_training_viability()
if viability["viable"]:
    print(f"\n[OK] 配置可训练：{viability['reason']}")
else:
    print(f"\n[无法训练] {viability['reason']}")
    print("  → 请改用带 NVIDIA GPU 或 Apple Silicon 的机器，或使用云端 GPU 实例")
    print("    （云端请用 hanzigen_cloudstudio.ipynb / hanzigen_colab.ipynb / hanzigen_moda.ipynb）")
    print("  → 数据准备（Cell 2）与 SVG 转换（Cell 5 后半）仍可在此机器运行，但训练（Cell 3/4）无法执行。")

# ===== 断连自检 + 字体切换检测 + 自动续训改写 =====
print("\n===== 断连自检 + 字体切换检测 =====")
os.makedirs("checkpoints", exist_ok=True)

def load_state() -> dict:
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_state(state: dict) -> None:
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

state = load_state()

# 字体切换检测：数据集状态与字体名绑定，防止新旧字体数据混合
prev_data_font = state.get("data_font")
if prev_data_font and prev_data_font != FONT_NAME:
    print(f"  [字体切换] {prev_data_font} → {FONT_NAME}")
    print("    · Cell 2 将重新执行数据准备（旧字体图像会被自动删除）")
    state["data_prep_done"] = False
    state["data_font"] = None

state.setdefault("font", FONT_NAME)
save_state(state)

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"
ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"
have_vqvae = os.path.exists(vqvae_ckpt)
have_ldm = os.path.exists(ldm_ckpt)

print(f"  数据准备: {'已完成' if state.get('data_prep_done') else '未完成'}（绑定字体: {state.get('data_font') or '无'}）")
print(f"  VQ-VAE 检查点: {'存在' if have_vqvae else '不存在'}")
print(f"  LDM 检查点:    {'存在' if have_ldm else '不存在'}")

# 自动精确续训：改写 local 脚本的 RESUME_FROM
def _set_resume_from(sh_path: str, ckpt: str) -> None:
    with open(sh_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'RESUME_FROM="[^"]*"', f'RESUME_FROM="{ckpt}"', content)
    with open(sh_path, "w", encoding="utf-8") as f:
        f.write(content)

_set_resume_from("scripts/train_vqvae_local.sh", vqvae_ckpt if have_vqvae else "")
_set_resume_from("scripts/train_ldm_local.sh", ldm_ckpt if have_ldm else "")
print(f"  [续训] train_vqvae_local.sh RESUME_FROM -> {vqvae_ckpt if have_vqvae else '(空，从零训练)'}")
print(f"  [续训] train_ldm_local.sh    RESUME_FROM -> {ldm_ckpt if have_ldm else '(空，从零训练)'}")

# 数据准备脚本按 CPU 核数调渲染并行度
cpu_cores = info["cpu_cores"]
render_workers = max(2, min(16, cpu_cores))
def set_sh_var(sh_path, var, value):
    with open(sh_path, encoding="utf-8") as f:
        lines = f.readlines()
    for i, ln in enumerate(lines):
        if ln.startswith(var + "="):
            body = ln[len(var)+1:].rstrip("\n")
            tail = ""
            if "#" in body:
                tail = "  " + body[body.index("#"):]
            lines[i] = f"{var}={value}{tail}\n"
            break
    with open(sh_path, "w", encoding="utf-8") as f:
        f.writelines(lines)

set_sh_var("scripts/prepare_dataset.sh", "NUM_WORKERS", render_workers)
# 本地机若无 CUDA，extract_charset 切 cpu（该步无实际张量计算）
set_sh_var("scripts/extract_charset.sh", "DEVICE", '"cuda"' if info["gpu_available"] else '"cpu"')

# 训练脚本（本地 conservative 档）
VQVAE_TRAIN_SCRIPT = "scripts/train_vqvae_local.sh"
LDM_TRAIN_SCRIPT = "scripts/train_ldm_local.sh"

print(f"\n[训练脚本] VQ-VAE → {VQVAE_TRAIN_SCRIPT}（conservative 本地档）")
print(f"[训练脚本] LDM     → {LDM_TRAIN_SCRIPT}（conservative 本地档）")
print("\n全部初始化完成！")

---
## Cell 2: 数据准备（纯 CPU，本地可直接运行）

> 分析字体覆盖率 → 渲染字形图片 → 提取训练/验证字符集。幂等，已完成会自动跳过。

In [ ]:
import os, json, subprocess

if not DO_DATA_PREP:
    print("DO_DATA_PREP=False，跳过 Cell 2")
else:
    def _load_state() -> dict:
        if os.path.exists(STATE_FILE):
            try:
                with open(STATE_FILE, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return {}
        return {}

    def _save_state(s: dict) -> None:
        with open(STATE_FILE, "w", encoding="utf-8") as f:
            json.dump(s, f, ensure_ascii=False, indent=2)

    state = _load_state()
    data_done = os.path.isdir("data/reference") and os.path.isdir("data/target")
    splits_done = os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt") and \
                  os.path.exists(f"charsets/splits/{FONT_NAME}/val.txt")
    same_font = state.get("data_font") == FONT_NAME

    if state.get("data_prep_done") and same_font and data_done and splits_done:
        print(f"字体 {FONT_NAME} 的数据准备已完成，跳过 Cell 2")
    else:
        print("\n===== 1. 分析字体覆盖率 =====")
        subprocess.run(["bash", "scripts/analyze_font.sh"], check=True)
        print("\n===== 2. 生成数据集图片 =====")
        subprocess.run(["bash", "scripts/prepare_dataset.sh"], check=True)
        print("\n===== 3. 提取训练/验证字符集 =====")
        subprocess.run(["bash", "scripts/extract_charset.sh"], check=True)

        if not (os.path.isdir("data/reference") and os.path.isdir("data/target")):
            raise RuntimeError("data/ 目录生成失败，请检查 prepare_dataset.sh")
        if not os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt"):
            raise RuntimeError("train.txt 未生成，请检查 extract_charset.sh")

        state["data_prep_done"] = True
        state["data_font"] = FONT_NAME
        _save_state(state)
        print("\n===== 数据准备完成 =====")

---
## Cell 3: 训练 VQ-VAE（conservative 本地档，约 6-8 小时）

> 需要 NVIDIA GPU 或 Apple Silicon。断连/重启后重跑 Cell 0、Cell 1 即可从断点 epoch 精确续训。

In [ ]:
import os, subprocess

if not DO_TRAIN_VQVAE:
    print("DO_TRAIN_VQVAE=False，跳过 Cell 3")
elif not os.path.isdir("data"):
    print("data/ 不存在，请先运行 Cell 2 完成数据准备")
elif not check_training_viability()["viable"]:
    print("当前机器不满足训练条件（无 NVIDIA GPU / Apple Silicon），无法训练 VQ-VAE")
else:
    print(f"将执行训练脚本: {VQVAE_TRAIN_SCRIPT}（conservative 档，batch/workers 运行时自适应）")
    subprocess.run(["bash", VQVAE_TRAIN_SCRIPT], check=True)

---
## Cell 4: 训练 LDM（conservative 本地档，约 10-15 小时）

> 依赖 Cell 3 产物 `checkpoints/vqvae_{FONT_NAME}.pth`。

In [ ]:
import os, subprocess

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"

if not DO_TRAIN_LDM:
    print("DO_TRAIN_LDM=False，跳过 Cell 4")
elif not os.path.exists(vqvae_ckpt):
    print(f"{vqvae_ckpt} 不存在，请先完成 Cell 3 训练 VQ-VAE")
elif not check_training_viability()["viable"]:
    print("当前机器不满足训练条件（无 NVIDIA GPU / Apple Silicon），无法训练 LDM")
else:
    print("VQ-VAE 检查点确认：")
    print(f"  {vqvae_ckpt}")
    print(f"将执行训练脚本: {LDM_TRAIN_SCRIPT}（conservative 档，batch/workers 运行时自适应）")
    subprocess.run(["bash", LDM_TRAIN_SCRIPT], check=True)

---
## Cell 5: 推理生成 + 指标 + 转换 SVG

> 依赖 Cell 4 产物 `checkpoints/ldm_{FONT_NAME}.pth`。

In [ ]:
import os, subprocess

ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"

if not DO_INFERENCE:
    print("DO_INFERENCE=False，跳过 Cell 5")
elif not os.path.exists(ldm_ckpt):
    print(f"{ldm_ckpt} 不存在，请先完成 Cell 4 训练 LDM")
else:
    # 推理生成（关键步骤）
    print("\n===== 推理生成 =====")
    subprocess.run(["bash", "scripts/inference.sh"], check=True)

    # 计算评估指标（非关键，失败不阻断）
    print("\n===== 计算评估指标 =====")
    try:
        subprocess.run(["bash", "scripts/compute_metrics.sh"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"[WARN] 评估指标计算失败（返回码 {e.returncode}），不影响补字结果。")

    # 转换 SVG
    print("\n===== 转换 SVG =====")
    subprocess.run(["bash", "scripts/convert_to_svg.sh"], check=True)
    print("\n===== 全部完成！SVG 输出在 svgs_{FONT_NAME}/ 目录 ====")

---
## 常见问题

| 问题 | 处理 |
|---|---|
| Cell 1 提示「无法训练」 | 你的机器无 NVIDIA GPU / Apple Silicon（如仅 AMD 核显）。训练需改用云端 GPU 实例，数据准备与 SVG 转换仍可本地跑 |
| 训练 OOM | 手动调低 `scripts/train_vqvae_local.sh` / `train_ldm_local.sh` 的 `BATCH_SIZE`（把 `auto` 改为具体整数） |
| 想压榨性能 | 把 local 脚本的 `PRESET=conservative` 改为 `PRESET=aggressive`（会占用更多 CPU，适合专用训练机） |
| 换字体后想重训 | 删除 `checkpoints/vqvae_{FONT_NAME}.pth`、`ldm_{FONT_NAME}.pth` 与 `colab_state.json`，重跑 Cell 1 |